In [ ]:
import os

# Vast.ai HF credential location
os.environ["HF_HOME"] = "/workspace/.hf_home"

from huggingface_hub import login, get_token

hf_token = get_token()

if hf_token:
    print("Using cached Hugging Face authentication.")
else:
    raise RuntimeError("Hugging Face token not found.")


In [ ]:
# Install specific compatible versions
import sys

!{sys.executable} -m pip install --upgrade pip setuptools wheel
!{sys.executable} -m pip install numpy scikit-learn


!{sys.executable} -m pip install git+https://github.com/AIPHES/emnlp19-moverscore.git

!{sys.executable} -m pip install accelerate rouge-score bert-score nltk torch torchvision pyemd datasets transformers peft "torchao>=0.16.0"

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)

# Verify
try:
    from moverscore_v2 import get_idf_dict, word_mover_score
    print("✓ MoverScore installed successfully")
except Exception as e:
    print(f"✗ Error: {e}")

In [ ]:
import nltk; nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)

import torch
import gc
import math
import os
import threading
import time
import json
from typing import Dict, Optional, Tuple
from huggingface_hub import hf_hub_download
from PIL import Image
from io import BytesIO
import requests


# Patch for numpy.float deprecation in MoverScore
import numpy as np
if not hasattr(np, 'float'):
    np.float = float

import re
import numpy as np
from transformers import EvalPrediction
import nltk
nltk.download('punkt', quiet=True)

from datasets import load_dataset
from transformers import (
    MllamaForConditionalGeneration,
    AutoProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
    set_seed,
)
from moverscore_v2 import get_idf_dict, word_mover_score
from peft import AdaLoraConfig, get_peft_model

SEED = 42
OUTPUT_DIR = os.path.abspath(".")

# Keys the collator puts in the batch that are metadata, NOT model inputs.
# Mllama forwards unknown **kwargs into the text model, so it must be
# stripped before every forward / generate call.
NON_MODEL_KEYS = ("prompt_length",)


class LazyImageCache:
    """Thread-safe cache for loaded images to avoid re-downloading."""
    def __init__(self, max_size: int = 100):
        self.cache = {}
        self.max_size = max_size
        self.lock = threading.Lock()

    def get(self, key: str) -> Optional[Image.Image]:
        with self.lock:
            return self.cache.get(key)

    def put(self, key: str, image: Image.Image):
        with self.lock:
            if len(self.cache) >= self.max_size:
                oldest_key = next(iter(self.cache))
                del self.cache[oldest_key]
            self.cache[key] = image

    def clear(self):
        with self.lock:
            self.cache.clear()


_image_cache = LazyImageCache()


def load_image_lazy(image_filename: str,
                   repo_id: str = "ChongyanChen/VQAonline",
                   cache_dir: str = "./vqa_images_cache") -> Image.Image:
    """Load image on-demand with caching to avoid repeated downloads."""
    cached_img = _image_cache.get(image_filename)
    if cached_img is not None:
        return cached_img.copy()

    if not isinstance(image_filename, str):
        if hasattr(image_filename, 'convert'):
            return image_filename.convert('RGB')
        else:
            placeholder = Image.new('RGB', (224, 224), (128, 128, 128))
            return placeholder

    filename = image_filename.strip()
    if not filename:
        return Image.new('RGB', (224, 224), (128, 128, 128))

    if os.path.exists(filename):
        try:
            img = Image.open(filename).convert('RGB')
            _image_cache.put(filename, img)
            return img.copy()
        except Exception:
            pass

    folders_to_try = [f"images{i}" for i in range(1, 8)]
    for folder in folders_to_try:
        try:
            file_path = f"{folder}/{filename}"
            local_path = hf_hub_download(
                repo_id=repo_id,
                filename=file_path,
                cache_dir=cache_dir,
                repo_type="dataset"
            )

            img = Image.open(local_path)
            img.verify()
            img = Image.open(local_path).convert('RGB')
            _image_cache.put(filename, img)
            return img.copy()
        except Exception:
            continue

    print(f"[WARN] Could not find image {filename}, using placeholder")
    placeholder = Image.new('RGB', (224, 224), (128, 128, 128))
    return placeholder


def load_image_from_source(image_source):
    """
    Robust image loading that handles strings, PIL Images, URLs, and file paths.

    Args:
        image_source: Can be a PIL Image, file path string, URL string, or base64 string

    Returns:
        PIL.Image in RGB mode or None if loading fails
    """
    try:
        if isinstance(image_source, Image.Image):
            return image_source.convert('RGB') if image_source.mode != 'RGB' else image_source

        if isinstance(image_source, str):
            if image_source.startswith(('http://', 'https://')):
                response = requests.get(image_source, timeout=10)
                response.raise_for_status()
                image = Image.open(BytesIO(response.content))
                return image.convert('RGB') if image.mode != 'RGB' else image

            if os.path.exists(image_source):
                image = Image.open(image_source)
                return image.convert('RGB') if image.mode != 'RGB' else image

            try:
                import base64
                image_data = base64.b64decode(image_source)
                image = Image.open(BytesIO(image_data))
                return image.convert('RGB') if image.mode != 'RGB' else image
            except Exception:
                pass

        if isinstance(image_source, bytes):
            image = Image.open(BytesIO(image_source))
            return image.convert('RGB') if image.mode != 'RGB' else image

        return None

    except Exception as e:
        print(f"Failed to load image: {e}")
        return None




class ReproducibilityTracker:
    """
    Tracks training reproducibility / efficiency metrics.

    Step latency is measured from on_step_begin to on_step_end,
    corresponding to one optimizer step including gradient accumulation.
    """

    def __init__(self):
        self.step_times = []
        self.training_start_time = None
        self.training_end_time = None
        self._step_start_time = None
        self.peak_memory_allocated = 0.0
        self.peak_memory_reserved = 0.0

    def start(self):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()

        self.training_start_time = time.perf_counter()

    def start_step(self):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        self._step_start_time = time.perf_counter()

    def end_step(self):
        if self._step_start_time is None:
            return

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        elapsed = time.perf_counter() - self._step_start_time

        if elapsed > 0:
            self.step_times.append(elapsed)

        self._step_start_time = None

    def stop(self):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        self.training_end_time = time.perf_counter()

        if torch.cuda.is_available():
            self.peak_memory_allocated = (
                torch.cuda.max_memory_allocated() / (1024 ** 3)
            )
            self.peak_memory_reserved = (
                torch.cuda.max_memory_reserved() / (1024 ** 3)
            )

    def summary(self):
        if (
            self.training_start_time is None
            or self.training_end_time is None
        ):
            return {}

        total_time = (
            self.training_end_time
            - self.training_start_time
        )

        if self.step_times:
            # Exclude first step because of CUDA/kernel initialization.
            measured_steps = (
                self.step_times[1:]
                if len(self.step_times) > 1
                else self.step_times
            )

            mean_ms = float(np.mean(measured_steps)) * 1000
            p99_ms = float(
                np.percentile(measured_steps, 99)
            ) * 1000
        else:
            mean_ms = 0.0
            p99_ms = 0.0

        return {
            "training_time_sec": total_time,
            "training_time_min": total_time / 60,
            "training_time_hours": total_time / 3600,
            "num_optimizer_steps": len(self.step_times),
            "mean_step_ms": mean_ms,
            "p99_step_ms": p99_ms,
            "peak_gpu_memory_allocated_gb": self.peak_memory_allocated,
            "peak_gpu_memory_reserved_gb": self.peak_memory_reserved,
        }


class ReproducibilityCallback(TrainerCallback):
    def __init__(self, tracker):
        self.tracker = tracker

    def on_train_begin(self, args, state, control, **kwargs):
        self.tracker.start()

    def on_step_begin(self, args, state, control, **kwargs):
        self.tracker.start_step()

    def on_step_end(self, args, state, control, **kwargs):
        self.tracker.end_step()

    def on_train_end(self, args, state, control, **kwargs):
        self.tracker.stop()



class AdaLoRARankAllocationCallback(TrainerCallback):
    """
    Drives AdaLoRA's rank allocation, which the HF Trainer never does on its own.

    PEFT requires model.base_model.update_and_allocate(step) after
    optimizer.step() and BEFORE zero_grad() (the importance scores use the
    gradients). Trainer fires on_optimizer_step exactly there, with
    state.global_step still 0-based — matching PEFT's reference loop.
    Without this, AdaLoRA never prunes anything and is just an
    SVD-parameterised LoRA at init_r with orthogonality regularisation.
    """

    def __init__(self, peft_model, output_dir):
        self.peft_model = peft_model
        self.output_dir = output_dir
        self.calls = 0

    def on_optimizer_step(self, args, state, control, **kwargs):
        # PEFT's importance update does p * p.grad for every adapter param and
        # crashes on grad=None (e.g. a tower that got no gradient this step);
        # treat those as zero-importance instead.
        for n, p in self.peft_model.named_parameters():
            if p.requires_grad and "lora_" in n and p.grad is None:
                p.grad = torch.zeros_like(p)
        self.peft_model.base_model.update_and_allocate(state.global_step)
        self.calls += 1

    def on_train_end(self, args, state, control, **kwargs):
        print(f"\nAdaLoRA: update_and_allocate called {self.calls} times")
        try:
            cfg = self.peft_model.peft_config["default"]
            pattern = cfg.rank_pattern or {}
            ranks = {name: int(sum(bool(x) for x in mask)) for name, mask in pattern.items()}
            if not ranks:
                print("AdaLoRA: no rank pattern recorded yet (budgeting phase not reached).")
                return
            vis = [r for n, r in ranks.items() if "vision_model" in n]
            lang = [r for n, r in ranks.items() if "language_model" in n]
            avg = lambda xs: (sum(xs) / len(xs)) if xs else 0.0
            print(f"AdaLoRA final avg rank -> all: {avg(list(ranks.values())):.3f} | "
                  f"vision: {avg(vis):.3f} | language: {avg(lang):.3f} (init_r={cfg.init_r}, target_r={cfg.target_r})")
            path = os.path.join(self.output_dir, "adalora_rank_pattern.json")
            with open(path, "w") as f:
                json.dump(ranks, f, indent=2)
            print(f"AdaLoRA rank pattern saved to {path}")
        except Exception as e:
            print(f"AdaLoRA: could not summarise rank pattern: {e}")


class SimpleListDataset(torch.utils.data.Dataset):
    """
    Holds the RAW streamed examples (PIL image + caption) and tokenizes lazily.

    The previous version materialised every TOKENIZED example
    (Dataset.from_list(list(...))), i.e. 4000 Mllama pixel tensors
    ([1, 4, 3, 560, 560] fp32, ~15 MB each, ~60 GB). Raw PIL images are far
    smaller. If an example fails to tokenize, the next valid one is returned.
    """

    def __init__(self, examples, tokenize_fn):
        self.examples = examples
        self.tokenize_fn = tokenize_fn

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        n = len(self.examples)
        for off in range(n):
            out = self.tokenize_fn(self.examples[(idx + off) % n])
            if out is not None:
                if off > 0:
                    print(f"[SimpleListDataset] example {idx % n} failed; substituted {(idx + off) % n}")
                return out
        raise RuntimeError("No example in this split could be tokenized.")


class AdaLoRAConfig:
    """Configuration for AdaLoRA training on Llama 3.2 11B Vision Instruct (image captioning, MS-COCO)."""

    def __init__(self):
        self.model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
        self.batch_size = 1
        self.gradient_accumulation_steps = 4
        self.num_epochs = 3
        self.learning_rate = 2e-6
        self.precision = "bf16"
        self.max_length = 1024

        # Split sizes: 8000 / 1000 / 1000 (train / val / held-out test)
        self.train_size = 8000
        self.val_size = 1000
        self.test_size = 1000

        # LoRA configuration
        self.lora_rank = 8
        self.lora_alpha = 16
        self.lora_dropout = 0.05

        # AdaLoRA budget. NOTE: AdaLoRA IGNORES `r` (lora_rank above) — the
        # ranks come from init_r / target_r. Every matrix starts at init_r and
        # is pruned to an AVERAGE of target_r. target_r = lora_rank keeps the
        # final budget equal to the rank-4 baselines; init_r uses PEFT's
        # default 1.5x headroom (PEFT default is 12 -> 8).
        self.adalora_target_r = self.lora_rank
        self.adalora_init_r = int(round(1.5 * self.lora_rank))
        self.adalora_tinit = 0       # warmup steps before pruning starts (PEFT default)
        self.adalora_tfinal = 0      # final fixed-rank steps (PEFT default)
        self.adalora_deltaT = 1      # allocate every optimizer step
        self.adalora_orth_reg_weight = 0.5

        # Llama Vision target modules: a plain list matches q/k/v in BOTH towers
        # (unchanged from the previous notebook).
        self.lora_target_modules = [
            "q_proj",
            "k_proj",
            "v_proj"
        ]
        self.apply_lora_to_vision = True
        self.vision_lora_modules = [
            "q_proj",
            "k_proj",
            "v_proj"
        ]

        # Dataset configuration
        self.dataset_name = "bitmind/MS-COCO"
        self.num_workers = 0
        self.pin_memory = False
        self.drop_last = True

        self.answer_max_length = 128
        self.max_caption_length = 128
        self.ignore_index = -100

        # Eval generation cap (unchanged: 64; generation stops at <|eot_id|>)
        self.generation_max_new_tokens = 64


def main():
    # Seed BEFORE the adapter is created, so AdaLoRA's A/B init is
    # reproducible (Trainer only seeds itself later, in __init__).
    set_seed(SEED)
    config = AdaLoRAConfig()

    print("Loading processor and model for AdaLoRA Llama Vision (image captioning)...")
    processor = AutoProcessor.from_pretrained(
        config.model_id,
        token=hf_token,
        trust_remote_code=True,
        padding_side="left",
    )
    tokenizer = processor.tokenizer

    if tokenizer.pad_token is None:
        # Llama-3 ships a dedicated pad token; fall back to EOS only if absent.
        if "<|finetune_right_pad_id|>" in tokenizer.get_vocab():
            tokenizer.pad_token = "<|finetune_right_pad_id|>"
        else:
            tokenizer.pad_token = tokenizer.eos_token
    print(f"Pad token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")

    # Stop generation on any Llama-3 end marker (instruct turns end with <|eot_id|>).
    eos_token_ids = []
    for t in [tokenizer.eos_token, "<|eot_id|>", "<|end_of_text|>"]:
        if t is None:
            continue
        tid = tokenizer.convert_tokens_to_ids(t)
        if isinstance(tid, int) and tid >= 0 and tid != tokenizer.unk_token_id and tid not in eos_token_ids:
            eos_token_ids.append(tid)
    print(f"EOS token ids for generation: {eos_token_ids}")

    model = MllamaForConditionalGeneration.from_pretrained(
        config.model_id,
        token=hf_token,
        torch_dtype=torch.bfloat16,
        attn_implementation="sdpa",
    )

    from datasets import load_dataset

    print("Loading dataset with streaming...")
    train_ds = load_dataset(config.dataset_name, split="train", streaming=True, cache_dir="./ic_cache")
    val_ds = load_dataset(config.dataset_name, split="validation", streaming=True, cache_dir="./ic_cache")
    test_ds = load_dataset(config.dataset_name, split="test", streaming=True, cache_dir="./ic_cache")

    # Materialise the RAW streamed examples (PIL image + caption); tokenization
    # happens lazily per item (see SimpleListDataset).
    train_examples = list(train_ds.take(config.train_size))
    val_examples = list(val_ds.take(config.val_size))
    test_examples = list(test_ds.take(config.test_size))

    # (previously a hardcoded print of "Train: 32, Val: 4, Test: 4")
    print(f"Final splits → Train: {len(train_examples)}, Val: {len(val_examples)}, Test: {len(test_examples)}")

    _debug_state = {"printed": False}

    def tokenize_vqa(example):
        """
        Llama / Mllama multimodal tokenization for image captioning.

        FIX (the previous version): it tokenized ONLY the prompt and pasted
        the caption ids into the last positions of `labels`, so the caption
        never appeared in input_ids — the model was trained to predict
        caption tokens at the positions of the chat-template tokens
        ("<|start_header_id|>assistant<|end_header_id|>\n\n"). That is the
        same bug the Gemma notebooks had and fixed.

        Now:
          1) prompt_text = chat template of [user: <image> + "Caption:"],
             add_generation_prompt=True (ends with the assistant header).
          2) full_text   = chat template of [user ..., assistant: caption]
             (ends with caption + <|eot_id|>).
          3) ONE processor call on (image, full_text) -> input_ids,
             attention_mask, pixel_values, aspect_ratio_ids,
             aspect_ratio_mask, cross_attention_mask, all consistent.
          4) prompt_length P = len(tokenized prompt_text), asserted to be an
             exact prefix; labels = -100 on [:P], caption (+ <|eot_id|>) on [P:].
        """
        try:
            if 'image' not in example or example['image'] is None:
                return None

            image = example['image']
            if not isinstance(image, Image.Image):
                image = load_image_from_source(image)
            if image is None or not isinstance(image, Image.Image):
                return None
            if image.mode != 'RGB':
                image = image.convert('RGB')

            caption = (example["sentences"]["raw"] or "").strip()
            # Cap the caption at max_caption_length tokens (as before).
            answer = tokenizer.decode(
                tokenizer.encode(caption, add_special_tokens=False)[:config.max_caption_length],
                skip_special_tokens=True,
            ).strip()

            if not answer:
                return None

            # Same instruction as the previous notebook.
            text_content = "Caption:"

            user_turn = {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": text_content},
                ],
            }
            assistant_turn = {
                "role": "assistant",
                "content": [{"type": "text", "text": answer}],
            }

            prompt_text = processor.apply_chat_template(
                [user_turn], add_generation_prompt=True, tokenize=False
            )
            full_text = processor.apply_chat_template(
                [user_turn, assistant_turn], add_generation_prompt=False, tokenize=False
            )

            if "<|image|>" not in prompt_text:
                print("ERROR: <|image|> token missing from chat template output.")
                return None

            full = processor(
                images=[[image]],
                text=[full_text],
                add_special_tokens=False,
                return_tensors="pt",
            )

            input_ids = full["input_ids"][0].long()
            attention_mask = full["attention_mask"][0].long()

            prompt_ids = tokenizer(
                prompt_text, add_special_tokens=False, return_tensors="pt"
            )["input_ids"][0].long()
            P = int(prompt_ids.shape[0])

            if P >= input_ids.shape[0] or not torch.equal(input_ids[:P], prompt_ids):
                print("WARNING: prompt ids are not a prefix of full ids — skipping example.")
                return None

            if input_ids.shape[0] > config.max_length:
                return None

            labels = input_ids.clone()
            labels[:P] = config.ignore_index

            cross_attention_mask = full["cross_attention_mask"][0]  # [seq, n_images, n_tiles]
            if cross_attention_mask.shape[0] != input_ids.shape[0]:
                print("WARNING: cross_attention_mask / input_ids length mismatch — skipping example.")
                return None

            output = {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels,
                "cross_attention_mask": cross_attention_mask,
                "pixel_values": full["pixel_values"][0],            # [n_images, n_tiles, 3, 560, 560]
                "aspect_ratio_ids": full["aspect_ratio_ids"][0],    # [n_images]
                "aspect_ratio_mask": full["aspect_ratio_mask"][0],  # [n_images, n_tiles]
                # metadata (stripped before the model sees the batch)
                "prompt_length": P,
            }

            if not _debug_state["printed"]:
                _debug_state["printed"] = True
                tgt = input_ids[P:]
                print("\n[Tokenization Debug] ---------------------------------")
                print(f"Full seq len: {input_ids.shape[0]} | prompt len: {P} | target len: {tgt.shape[0]}")
                print(f"Prompt tail : {tokenizer.decode(input_ids[max(0, P - 8):P])!r}")
                print(f"Target (raw): {tokenizer.decode(tgt)!r}")
                print(f"pixel_values {tuple(output['pixel_values'].shape)} | "
                      f"aspect_ratio_ids {tuple(output['aspect_ratio_ids'].shape)} | "
                      f"aspect_ratio_mask {tuple(output['aspect_ratio_mask'].shape)} | "
                      f"cross_attention_mask {tuple(cross_attention_mask.shape)}")
                print("-----------------------------------------------------\n")

            return output

        except Exception as e:
            print(f"Error processing example: {e}")
            import traceback
            traceback.print_exc()
            return None

    print("Building lazy datasets (on-the-fly tokenization)...")
    tokenized_train_dataset = SimpleListDataset(train_examples, tokenize_vqa)
    tokenized_val_dataset = SimpleListDataset(val_examples, tokenize_vqa)
    tokenized_test_dataset = SimpleListDataset(test_examples, tokenize_vqa)

    train_len = len(tokenized_train_dataset)
    val_len = len(tokenized_val_dataset)
    test_len = len(tokenized_test_dataset)

    if train_len == 0:
        raise ValueError(
            "No training samples after processing. "
            "Possible causes: images not loading correctly, dataset format incompatible, "
            "or all examples failed validation."
        )

    # Fail fast: tokenize one example now (also prints the debug block above).
    _probe = tokenized_train_dataset[0]
    assert _probe is not None

    eff_batch = config.batch_size * config.gradient_accumulation_steps
    total_steps = math.ceil(train_len / eff_batch) * config.num_epochs

    target_modules = config.lora_target_modules.copy()

    # AdaLoRA applied ONCE, with the real step count.
    lora_config = AdaLoraConfig(
        init_r=config.adalora_init_r,
        target_r=config.adalora_target_r,
        tinit=config.adalora_tinit,
        tfinal=config.adalora_tfinal,
        deltaT=config.adalora_deltaT,
        orth_reg_weight=config.adalora_orth_reg_weight,
        total_step=total_steps,
        lora_alpha=config.lora_alpha,
        target_modules=target_modules,
        lora_dropout=config.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        inference_mode=False,
        use_rslora=True,
    )

    model = get_peft_model(model, lora_config)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.enable_input_require_grads()
    model.config.use_cache = False
    if getattr(model.config, "text_config", None) is not None:
        model.config.text_config.use_cache = False
    meta_params = [
        name for name, param in model.named_parameters()
        if param.device.type == "meta"
    ]

    if meta_params:
        raise RuntimeError(
            f"Model still contains {len(meta_params)} meta parameters. "
            f"First few: {meta_params[:10]}"
        )

    print("✓ Model has no meta parameters")
    print(f"✓ Model device: {next(model.parameters()).device}")
    from peft.tuners.lora import LoraLayer

    vision_modules = 0
    language_modules = 0

    for name, module in model.named_modules():
        if isinstance(module, LoraLayer) and module.r.get('default', 0) > 0:
            name_lower = name.lower()

            is_vision = any([
                'vision_tower' in name_lower,
                'vision_model' in name_lower,
                'vision_encoder' in name_lower,
                'visual' in name_lower and 'language' not in name_lower,
                name.startswith('model.vision'),
                name.startswith('vision_tower'),
            ])

            if is_vision:
                vision_modules += 1
            else:
                language_modules += 1

    print(
        f"Instrumented {vision_modules} vision modules + "
        f"{language_modules} language modules"
    )
    print(f"Total LoRA modules: {vision_modules + language_modules}")

    trainable_params = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )

    total_params = sum(
        p.numel() for p in model.parameters()
    )

    print(f"Trainable params: {trainable_params:,}")
    print(f"Total params: {total_params:,}")
    print(f"Percentage: {100 * trainable_params / total_params:.3f}%")

    print(f"\nSuccessfully processed datasets:")
    print(f"Training samples: {train_len}")
    print(f"Validation samples: {val_len}")
    print(f"Test samples: {test_len}")

    print(f"\nTraining Configuration:")
    print(f"Total training steps: {total_steps}")
    print(f"Effective batch size: {eff_batch}")
    print(f"Steps per epoch: {math.ceil(train_len / eff_batch)}")

    training_args = Seq2SeqTrainingArguments(
        output_dir="./results_ic_adalora_llama4_seed42",

        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,

        num_train_epochs=config.num_epochs,
        learning_rate=config.learning_rate,

        bf16=(config.precision == "bf16"),
        fp16=(config.precision == "fp16"),

        max_grad_norm=0.5,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},

        dataloader_num_workers=config.num_workers,
        dataloader_drop_last=config.drop_last,
        dataloader_pin_memory=config.pin_memory,

        remove_unused_columns=False,
        report_to="none",

        # Final training: no validation/checkpointing during training
        save_strategy="no",
        eval_strategy="no",

        logging_steps=50,
        warmup_steps=200,
        optim="adamw_torch",

        # No best-checkpoint selection because evaluation is disabled
        load_best_model_at_end=False,

        seed=SEED,
        prediction_loss_only=False,
        predict_with_generate=True,
    )

    pixel_dtype = torch.bfloat16 if config.precision == "bf16" else (
        torch.float16 if config.precision == "fp16" else torch.float32
    )

    def _pad_tiles(t, target_tiles, tile_dim):
        cur = t.shape[tile_dim]
        if cur == target_tiles:
            return t
        pad_shape = list(t.shape)
        pad_shape[tile_dim] = target_tiles - cur
        return torch.cat([t, t.new_zeros(pad_shape)], dim=tile_dim)

    def collate_fn(batch):
        """
        Mllama collator.
        - RIGHT-pads input_ids / attention_mask / labels (so prediction_step
          can take the prompt as input_ids[i, :prompt_length]).
        - Pads cross_attention_mask along the sequence dim with zeros so it
          always has exactly the same length as input_ids.
        - Stacks the per-image vision tensors.
        """
        batch = [b for b in batch if b is not None and isinstance(b, dict) and 'input_ids' in b]
        if len(batch) == 0:
            raise RuntimeError("Empty batch in collate_fn.")

        pad_id = tokenizer.pad_token_id

        input_ids = torch.nn.utils.rnn.pad_sequence(
            [b['input_ids'] for b in batch], batch_first=True, padding_value=pad_id
        )
        attention_mask = torch.nn.utils.rnn.pad_sequence(
            [b['attention_mask'] for b in batch], batch_first=True, padding_value=0
        )
        labels = torch.nn.utils.rnn.pad_sequence(
            [b['labels'] for b in batch], batch_first=True, padding_value=config.ignore_index
        )
        max_len = input_ids.shape[1]

        # The processor pads to max_image_tiles already; this is just a guard.
        max_tiles = max(b['pixel_values'].shape[1] for b in batch)

        cams, pvs, ar_ids, ar_masks = [], [], [], []
        for b in batch:
            cam = _pad_tiles(b['cross_attention_mask'], max_tiles, tile_dim=-1)
            seq_pad = max_len - cam.shape[0]
            if seq_pad > 0:
                cam = torch.cat([cam, cam.new_zeros((seq_pad,) + tuple(cam.shape[1:]))], dim=0)
            cams.append(cam)
            pvs.append(_pad_tiles(b['pixel_values'], max_tiles, tile_dim=1))
            ar_ids.append(b['aspect_ratio_ids'])
            ar_masks.append(_pad_tiles(b['aspect_ratio_mask'], max_tiles, tile_dim=-1))

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'cross_attention_mask': torch.stack(cams, dim=0),          # [B, seq, n_img, n_tiles]
            'pixel_values': torch.stack(pvs, dim=0).to(pixel_dtype),  # [B, n_img, n_tiles, 3, H, W]
            'aspect_ratio_ids': torch.stack(ar_ids, dim=0),            # [B, n_img]
            'aspect_ratio_mask': torch.stack(ar_masks, dim=0),         # [B, n_img, n_tiles]
            # metadata — popped / stripped before the model sees the batch
            'prompt_length': torch.tensor([int(b['prompt_length']) for b in batch], dtype=torch.long),
        }

    def make_compute_metrics(processor):
        import re
        import numpy as np
        from rouge_score import rouge_scorer
        from nltk.translate.meteor_score import meteor_score
        from bert_score import score as bert_score
        from collections import defaultdict
        import nltk
        nltk.download('wordnet', quiet=True);
        nltk.download('omw-1.4', quiet=True)

        # PATCH FOR MOVERSCORE COMPATIBILITY
        import transformers.tokenization_utils_base as tub
        if not hasattr(tub.PreTrainedTokenizerBase, '_moverscore_patched'):
            original_init = tub.PreTrainedTokenizerBase.__init__
            def patched_init(self, *args, **kwargs):
                original_init(self, *args, **kwargs)
                if not hasattr(self, 'max_len'):
                    self.max_len = self.model_max_length
            tub.PreTrainedTokenizerBase.__init__ = patched_init
            tub.PreTrainedTokenizerBase._moverscore_patched = True
            print("✓ Applied moverscore compatibility patch")

        # Now import moverscore
        moverscore_available = False
        try:
            from moverscore_v2 import get_idf_dict, word_mover_score
            moverscore_available = True
            print("✓ MoverScore loaded successfully")
        except ImportError as e:
            print(f"✗ MoverScore not available: {e}")

        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)


        def extract_answer(text):
            """Clean Llama-3 output safely without truncating content."""

            if text is None:
                return ""

            # Remove Llama-3 special tokens (normally already skipped by decode)
            text = text.replace("<|begin_of_text|>", "")
            text = text.replace("<|end_of_text|>", "")
            text = text.replace("<|start_header_id|>", "")
            text = text.replace("<|end_header_id|>", "")
            text = text.replace("<|eot_id|>", "")
            text = text.replace("<|image|>", "")

            # Remove an accidental leading assistant header if it survives decoding
            text = re.sub(r"^\s*assistant\s*", "", text, flags=re.IGNORECASE)

            # Normalize whitespace
            text = re.sub(r"\s+", " ", text).strip()

            return text

        def normalize(s):
            """VQAonline normalization."""
            s = re.sub(r"\s+", " ", re.sub(r"([.!?,;])", r" \1", s.lower().strip()))
            s = re.sub(r" n't ", " not ", s)
            s = re.sub(r" 's ", " is ", s)
            s = re.sub(r" 're ", " are ", s)
            s = re.sub(r" 've ", " have ", s)
            s = re.sub(r" 'll ", " will ", s)
            s = re.sub(r" 'd ", " would ", s)
            return s.strip()

        def compute_vqa_metrics(p: EvalPrediction):
            pred_ids = np.asarray(p.predictions, dtype=np.int64)
            labels = np.asarray(p.label_ids, dtype=np.int64)

            tok = processor.tokenizer

            def safe_ids(arr):
                """
                Llama-safe id cleanup before decoding.
                - negative ids (-100 label mask / Trainer's cross-batch
                  generation padding) -> pad id.
                - upper bound is len(tokenizer), NOT tokenizer.vocab_size:
                  clipping to vocab_size-1 turns <|eot_id|> (128009) into
                  token 127999, a real text token.
                """
                arr = np.asarray(arr, dtype=np.int64).copy()
                arr[arr < 0] = tok.pad_token_id
                arr[arr >= len(tok)] = tok.pad_token_id
                return arr

            try:
                pred_texts_raw = tok.batch_decode(safe_ids(pred_ids), skip_special_tokens=True)
                label_texts = tok.batch_decode(safe_ids(labels), skip_special_tokens=True)
            except Exception as e:
                print(f"[Metrics] Decode error: {e}")
                return {"rouge_l": 0.0, "meteor": 0.0, "bertscore_f1": 0.0, "moverscore": 0.0}

            pred_texts = [extract_answer(t) for t in pred_texts_raw]
            pred_norm = [normalize(t) for t in pred_texts]
            label_texts = [extract_answer(t) for t in label_texts]
            label_norm = [normalize(t) for t in label_texts]

            for i in range(min(3, len(pred_norm))):
                print(f"\n[Sample {i+1}]")
                print(f"Generated: {pred_texts[i][:100]}")
                print(f"Reference: {label_texts[i][:100]}")

            valid_preds = []
            valid_labels = []
            for p, l in zip(pred_norm, label_norm):
                if p and l and len(p.split()) > 0 and len(l.split()) > 0:
                    valid_preds.append(p)
                    valid_labels.append(l)

            # ROUGE-L
            rouge_ls = []
            for p, l in zip(pred_norm, label_norm):
                if p and l:
                    rouge_ls.append(scorer.score(l, p)['rougeL'].fmeasure)
            rouge_l = np.mean(rouge_ls) if rouge_ls else 0.0

            # METEOR
            meteors = []
            for p, l in zip(pred_norm, label_norm):
                if p and l:
                    meteors.append(meteor_score([l.split()], p.split()))
            meteor = np.mean(meteors) if meteors else 0.0

            # BERTscore
            if valid_preds and valid_labels:
                try:
                    P, R, F1 = bert_score(
                        valid_preds,
                        valid_labels,
                        lang="en",
                        model_type="bert-base-uncased",
                        rescale_with_baseline=False
                    )
                    bertscore_f1 = F1.mean().item()
                except Exception as e:
                    print(f"[Metrics] BERTScore error: {e}")
                    bertscore_f1 = 0.0
            else:
                bertscore_f1 = 0.0

            # MoverScore
            moverscore_val = 0.0
            if moverscore_available and valid_preds and valid_labels:
                try:
                    from moverscore_v2 import get_idf_dict, word_mover_score

                    idf_dict_ref = get_idf_dict(valid_labels)
                    idf_dict_hyp = get_idf_dict(valid_preds)

                    scores = word_mover_score(
                        valid_labels,
                        valid_preds,
                        idf_dict_ref,
                        idf_dict_hyp,
                        stop_words=[],
                        n_gram=1,
                        remove_subwords=True
                    )

                    if scores and len(scores) > 0:
                        moverscore_val = float(np.mean(scores))
                        print(f"[MoverScore] Score: {moverscore_val:.4f}")

                except Exception as e:
                    print(f"[MoverScore] Error: {e}")
                    import traceback
                    traceback.print_exc() # <--- Added this line

            return {
                "rouge_l": float(rouge_l),
                "meteor": float(meteor),
                "bertscore_f1": float(bertscore_f1),
                "moverscore": float(moverscore_val),
            }

        return compute_vqa_metrics

    # Custom Trainer with VRAM clearing before evaluation
    class LoRATrainer(Seq2SeqTrainer):
        """AdaLoRA Seq2SeqTrainer with Llama-safe metadata handling and prompt-only generation."""

        def __init__(self, processor, gen_config, eos_token_ids, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.processor = processor
            self.gen_config = gen_config  # AdaLoRAConfig instance
            # Llama-3 instruct turns end with <|eot_id|>, not the base EOS.
            self.eos_token_ids = eos_token_ids

        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            # Standard Trainer loss (AdaLoRA's orthogonal regulariser is added
            # inside the PEFT forward) — only real model inputs go to Mllama.
            model_inputs = {k: v for k, v in inputs.items() if k not in NON_MODEL_KEYS}
            return super().compute_loss(
                model, model_inputs, return_outputs=return_outputs, num_items_in_batch=num_items_in_batch
            )

        def evaluate(self, *args, **kwargs):
            print("\nClearing VRAM before evaluation...")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return super().evaluate(*args, **kwargs)

        def prediction_step(
            self,
            model: torch.nn.Module,
            inputs: Dict[str, torch.Tensor],
            prediction_loss_only: bool,
            ignore_keys: Optional[list] = None,
        ) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor], Optional[torch.Tensor]]:
            """
            Llama / Mllama version.

            inputs["input_ids"] is the FULL chat-templated sequence
                <|begin_of_text|> user header <|image|> instruction <|eot_id|>
                assistant header  answer <|eot_id|>
            produced by ONE processor call, so cross_attention_mask matches it
            position-for-position. The collator RIGHT-pads, so the prompt of
            example i is always input_ids[i, :prompt_length[i]].

            For generation we SLICE (never extend) input_ids, attention_mask AND
            cross_attention_mask to the prompt length, then left-repad. Slicing a
            processor-built mask gives exactly what a prompt-only processor call
            would give (tokenize_vqa asserts the prompt ids are an exact
            prefix of the full ids), so the text/mask lengths always agree ->
            no device-side assert. pixel_values / aspect_ratio_ids /
            aspect_ratio_mask are per-image and carried over unchanged.

            The teacher-forced eval loss still uses the full [prompt][caption]
            sequence.
            """
            labels = inputs.pop("labels", None)
            has_labels = labels is not None

            prompt_lengths = inputs.pop("prompt_length", None)

            inputs = self._prepare_inputs(inputs)
            input_ids = inputs["input_ids"]
            input_len = input_ids.shape[1]

            if prompt_lengths is not None:
                attn = inputs["attention_mask"]
                cam = inputs.get("cross_attention_mask", None)
                pad_id = self.processor.tokenizer.pad_token_id
                bsz = input_ids.shape[0]
                p_lens = [int(prompt_lengths[i]) for i in range(bsz)]
                max_p = max(p_lens)

                gen_input_ids = torch.full((bsz, max_p), pad_id, dtype=input_ids.dtype, device=input_ids.device)
                gen_attention_mask = torch.zeros((bsz, max_p), dtype=attn.dtype, device=attn.device)
                gen_cam = None
                if cam is not None:
                    gen_cam = torch.zeros((bsz, max_p) + tuple(cam.shape[2:]), dtype=cam.dtype, device=cam.device)

                # Re-pad on the LEFT for batched decoding.
                for i, p in enumerate(p_lens):
                    gen_input_ids[i, max_p - p:] = input_ids[i, :p]
                    gen_attention_mask[i, max_p - p:] = attn[i, :p]
                    if gen_cam is not None:
                        gen_cam[i, max_p - p:] = cam[i, :p]

                gen_inputs = {
                    "input_ids": gen_input_ids,
                    "attention_mask": gen_attention_mask,
                }
                if gen_cam is not None:
                    gen_inputs["cross_attention_mask"] = gen_cam
                for k in ("pixel_values", "aspect_ratio_ids", "aspect_ratio_mask"):
                    if k in inputs:
                        gen_inputs[k] = inputs[k]

                gen_input_len = gen_input_ids.shape[1]
            else:
                # Should not happen with tokenize_vqa; kept as a fallback.
                gen_inputs = {k: v for k, v in inputs.items() if k not in NON_MODEL_KEYS}
                gen_input_len = input_len

            with torch.no_grad():
                if prediction_loss_only:
                    loss = None
                    if has_labels:
                        inputs_copy = dict(inputs)
                        inputs_copy["labels"] = labels
                        loss = self.compute_loss(model, inputs_copy).detach()
                    return (loss, None, None)

                generated_tokens = model.generate(
                    **gen_inputs,
                    max_new_tokens=self.gen_config.generation_max_new_tokens,
                    do_sample=False,
                    use_cache=True,
                    pad_token_id=self.processor.tokenizer.pad_token_id,
                    eos_token_id=self.eos_token_ids,
                )

                # Decoder-only output = [prompt][new tokens]; keep new tokens only.
                generated_tokens = generated_tokens[:, gen_input_len:]

                # Teacher-forced loss on the FULL [prompt][answer] sequence.
                loss = None
                if has_labels:
                    inputs_copy = dict(inputs)
                    inputs_copy["labels"] = labels
                    loss = self.compute_loss(model, inputs_copy).detach()

            if labels is not None:
                if labels.dim() == 3:
                    labels = torch.argmax(labels, dim=-1)
                elif labels.dim() == 1:
                    batch_size = input_ids.shape[0]
                    labels = labels.view(batch_size, -1)

            return loss, generated_tokens, labels

    tracker = ReproducibilityTracker()

    trainer = LoRATrainer(
        processor=processor,
        gen_config=config,
        eos_token_ids=eos_token_ids,
        model=model,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_val_dataset,
        data_collator=collate_fn,
        compute_metrics=make_compute_metrics(processor),
        callbacks=[ReproducibilityCallback(tracker), AdaLoRARankAllocationCallback(model, OUTPUT_DIR)],
    )

    print("\n" + "="*80)
    print("STARTING ADALORA TRAINING FOR LLAMA 3.2 11B VISION (IMAGE CAPTIONING)")
    print("="*80)
    print(f"Model: {config.model_id}")
    print(f"Dataset: {config.dataset_name}")
    print(f"AdaLoRA init_r -> target_r: {config.adalora_init_r} -> {config.adalora_target_r} "
          f"(tinit={config.adalora_tinit}, tfinal={config.adalora_tfinal}, deltaT={config.adalora_deltaT}, total_step={total_steps})")
    print(f"Batch size: {config.batch_size} x {config.gradient_accumulation_steps} = {eff_batch}")
    print(f"Learning rate: {config.learning_rate}")
    print(f"Precision: {config.precision}")
    print(f"Training samples: {train_len}")
    print(f"Validation samples: {val_len}")
    print(f"Test samples: {test_len}")
    print("="*80 + "\n")

    try:
        print("Beginning training...")
        trainer.train()

        # Reproducibility / efficiency metrics
        repro_metrics = tracker.summary()
        total_optimizer_steps = trainer.state.global_step

        print("\n" + "="*80)
        print("REPRODUCIBILITY / EFFICIENCY METRICS")
        print("="*80)
        print(f"Training wall-clock : {repro_metrics['training_time_hours']:.4f} hours")
        print(f"Training wall-clock : {repro_metrics['training_time_min']:.2f} minutes")
        print(f"Optimizer steps     : {total_optimizer_steps}")
        print(f"Mean step time      : {repro_metrics['mean_step_ms']:.2f} ms")
        print(f"P99 step time       : {repro_metrics['p99_step_ms']:.2f} ms")
        print(f"Peak GPU allocated  : {repro_metrics['peak_gpu_memory_allocated_gb']:.2f} GB")
        print(f"Peak GPU reserved   : {repro_metrics['peak_gpu_memory_reserved_gb']:.2f} GB")
        print("="*80)

        test_results = trainer.evaluate(eval_dataset=tokenized_test_dataset)
        final_results = {
            "method": "AdaLoRA",
            "model": config.model_id,
            "task": "Image Captioning (MS-COCO)",
            "seed": SEED,
            "train_samples": train_len,
            "val_samples": val_len,
            "test_samples": test_len,
            "learning_rate": config.learning_rate,
            "lora_target_modules": config.lora_target_modules,
            "adalora_init_r": config.adalora_init_r,
            "adalora_target_r": config.adalora_target_r,
            "adalora_tinit": config.adalora_tinit,
            "adalora_tfinal": config.adalora_tfinal,
            "adalora_deltaT": config.adalora_deltaT,
            "adalora_orth_reg_weight": config.adalora_orth_reg_weight,
            "test_metrics": test_results,
            "reproducibility_metrics": {
                "training_wall_clock_hours": repro_metrics["training_time_hours"],
                "training_wall_clock_minutes": repro_metrics["training_time_min"],
                "optimizer_steps": total_optimizer_steps,
                "mean_step_ms": repro_metrics["mean_step_ms"],
                "p99_step_ms": repro_metrics["p99_step_ms"],
                "peak_gpu_memory_allocated_gb": repro_metrics[
                    "peak_gpu_memory_allocated_gb"
                ],
                "peak_gpu_memory_reserved_gb": repro_metrics[
                    "peak_gpu_memory_reserved_gb"
                ],
            }
        }

        with open("AdaLora_llama8_seed42_IC.json", "w") as f:
            json.dump(final_results, f, indent=2)

        print("Results saved to AdaLora_llama4_seed42_IC.json")

        print(f"Test Loss          : {test_results['eval_loss']:.5f}")
        print(f"Test ROUGE-L       : {test_results['eval_rouge_l']:.4f}")
        print(f"Test METEOR        : {test_results['eval_meteor']:.4f}")
        print(f"Test BERTscore F1  : {test_results['eval_bertscore_f1']:.4f}")
        print(f"Test MoverScore    : {test_results['eval_moverscore']:.4f}")

        print("\n" + "="*80)
        print("Saving final model...")
        output_dir = "./final_ic_adalora_llama4_seed42"
        model.save_pretrained(output_dir)
        processor.save_pretrained(output_dir)
        print(f"Model saved to {output_dir}")

        print("\n" + "="*80)
        print("TRAINING COMPLETED SUCCESSFULLY")
        print("="*80 + "\n")

    except KeyboardInterrupt:
        print("\n" + "="*80)
        print("TRAINING INTERRUPTED BY USER")
        print("="*80)
        print("Saving checkpoint...")
        try:
            model.save_pretrained("./interrupted_checkpoint_ic_adalora_llama4")
            processor.save_pretrained("./interrupted_checkpoint_ic_adalora_llama4")
            print("Checkpoint saved to ./interrupted_checkpoint_ic_adalora_llama4")
        except Exception as e:
            print(f"Failed to save checkpoint: {e}")

    except Exception as e:
        print("\n" + "="*80)
        print("TRAINING FAILED")
        print("="*80)
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        raise

    finally:
        print("\nCleaning up...")
        _image_cache.clear()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("Cleanup complete")


if __name__ == "__main__":
    main()
